A notebook to process LLM outputs into familiar formats.

## Data Loading

In [2]:

from dataset_processing import process_llm_jsonl_results, CLIRENER_LABELS_V1, transform_to_ner_format
from EXPERIMENTS.evaluate_gold import load_and_merge_gold_data
import wandb
from EXPERIMENTS.evaluate import run_nervaluate, log_to_wandb

Based on [`run_campaign()`](EXPERIMENTS/evaluate_gold.py):

In [3]:
# True data loading
GOLD_DATASET_ID = "P0L3/CliReNER_v_1_1_28_GOLD_authorannots"
gold_data, bio_label_list = load_and_merge_gold_data(GOLD_DATASET_ID)
target_labels = list(CLIRENER_LABELS_V1)

# Predictions loading
llm_predictions_dir = "RESULTS/LLM_PREDICTIONS/ner_results_gemini_3_pro_preview.jsonl"
llm_name = llm_predictions_dir.split("/")[-1].replace("ner_results_", "").replace(".jsonl", "")
raw_predictions = process_llm_jsonl_results(llm_predictions_dir)
print(llm_name)

# WANDB initialization
WANDB_PROJECT = "CLIRENER_GOLD_SEEDS_authorannots"

# B. Initialize WandB Run
run_name = f"eval_GOLD_{llm_name}_zs"

# # Start a fresh run for this evaluation
# run = wandb.init(
#     project=WANDB_PROJECT,
#     name=run_name,
#     reinit=True, # Allow multiple runs in one script
#     config={
#         "model_type": "LLM_"+llm_name,
#         "model_id": llm_name,
#         "training_dataset": "None",
#         "evaluation_dataset": GOLD_DATASET_ID, # Explicitly logging this
#         "seed": -1,
#         "evaluation_scope": "ALL_SPLITS_MERGED"
#     }
# )

# D. Transform Predictions
print("--- Transforming Predictions ---")
model_predictions_transformed = transform_to_ner_format(raw_predictions, target_labels)

pred_lookup = {}
# Iterate through all predictions (including duplicates/failed attempts in the JSONL)
for row in model_predictions_transformed[0]:
    # print(row)
    text = row['text']
    tags = row['ner_tags']
    
    # Check if tags are valid (not None) and not empty.
    # If a sentence was attempted twice (once failed, once success), 
    # this logic ensures we only store the successful one.
    if tags is not None and len(tags) > 0:
        pred_lookup[text] = tags

true_ids = []
pred_ids = []
missing_count = 0
mismatch_count = 0

for row in gold_data["test"]:
    text_key = row['text']
    
    # 3. Match by exact text
    if text_key in pred_lookup:
        p_tags = pred_lookup[text_key]
        
        # Additional safety: Ensure lengths match (LLM might have truncated text)
        if len(row['ner_tags']) == len(p_tags):
            true_ids.append(row['ner_tags'])
            pred_ids.append(p_tags)
        else:
            mismatch_count += 1
    else:
        missing_count += 1

if missing_count > 0:
    print(f"Warning: {missing_count} rows from Gold Data were not found in Predictions.")
if mismatch_count > 0:
    print(f"Warning: {mismatch_count} rows were ignored due to token length mismatch.")
    
print(f"aligned {len(true_ids)} rows for evaluation.")



# E. Calculate Metrics
# We pass the dataset's BIO scheme for ID mapping, and the target labels for reporting
results, results_by_tag = run_nervaluate(true_ids, pred_ids, bio_label_list, tags=target_labels)

# # F. Log to WandB
# log_to_wandb(results, results_by_tag)

print(f"SUCCESS: {run_name}")
# print(results)
# wandb.finish()

--- Loading and Merging GOLD Dataset: P0L3/CliReNER_v_1_1_28_GOLD_authorannots ---
Merged ['train', 'validation', 'test'] into a single dataset of size: 192
gemini_3_pro_preview
--- Transforming Predictions ---
aligned 135 rows for evaluation.
--- Calculating Metrics ---
SUCCESS: eval_GOLD_gemini_3_pro_preview_zs


In [3]:
results

{'ent_type': {'correct': 972,
  'incorrect': 393,
  'partial': 0,
  'missed': 193,
  'spurious': 153,
  'possible': 1558,
  'actual': 1518,
  'precision': 0.6403162055335968,
  'recall': 0.6238767650834403,
  'f1': 0.6319895968790638},
 'partial': {'correct': 1098,
  'incorrect': 0,
  'partial': 267,
  'missed': 193,
  'spurious': 153,
  'possible': 1558,
  'actual': 1518,
  'precision': 0.8112648221343873,
  'recall': 0.790436456996149,
  'f1': 0.8007152145643693},
 'strict': {'correct': 841,
  'incorrect': 524,
  'partial': 0,
  'missed': 193,
  'spurious': 153,
  'possible': 1558,
  'actual': 1518,
  'precision': 0.5540184453227931,
  'recall': 0.5397946084724005,
  'f1': 0.546814044213264},
 'exact': {'correct': 1098,
  'incorrect': 267,
  'partial': 0,
  'missed': 193,
  'spurious': 153,
  'possible': 1558,
  'actual': 1518,
  'precision': 0.7233201581027668,
  'recall': 0.7047496790757382,
  'f1': 0.7139141742522757}}